In [1]:
from torch.utils.data import DataLoader, TensorDataset
from metrics          import eval_model, compare_metric
from gensim.models    import Word2Vec
from model            import GenerateModel

import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy             as np
import torch
import os

device     = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu'); print(f'Deivce: {device}')
model_path = os.path.join(os.getcwd(),'Model','model.tar')

Deivce: mps


In [2]:
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')

Y_train = np.load('Y_train.npy')
Y_test  = np.load('Y_test.npy')

X_train = torch.from_numpy(X_train).long()
Y_train = torch.from_numpy(Y_train).type(torch.float32)
X_test  = torch.from_numpy(X_test) .long()
Y_test  = torch.from_numpy(Y_test) .type(torch.float32) 

train_loader = DataLoader(TensorDataset(X_train,Y_train), batch_size = 1,shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test ,Y_test) , batch_size = 64,shuffle=False)

X_sample, Y_samples = next(iter(train_loader))

model = GenerateModel()
model.load_state_dict(torch.load(model_path,map_location=torch.device('cpu'),weights_only=True))
model.eval()

ConvAttnPool(
  (embed): Embedding(150854, 100, padding_idx=150853)
  (conv): Conv1d(100, 15, kernel_size=(5,), stride=(1,), padding=(2,))
  (U): Linear(in_features=15, out_features=50, bias=True)
  (final): Linear(in_features=15, out_features=50, bias=True)
  (embed_drop): Dropout(p=0.2, inplace=False)
)

In [3]:
# Odd filter size
model = GenerateModel(num_of_filters=15,kernel_size=5)
model.eval()

ConvAttnPool(
  (embed): Embedding(150854, 100, padding_idx=150853)
  (conv): Conv1d(100, 15, kernel_size=(5,), stride=(1,), padding=(2,))
  (U): Linear(in_features=15, out_features=50, bias=True)
  (final): Linear(in_features=15, out_features=50, bias=True)
  (embed_drop): Dropout(p=0.2, inplace=False)
)

In [7]:
word_model = Word2Vec.load('processed_full.w2v')

In [97]:
print(f'X_train shape    => {X_sample.shape}')
x = model.embed(X_sample)
print(f'Embedding Shpae  => {x.shape}')
x = x.transpose (1, 2)
print(f'Transpose Shape  => {x.shape}')
x = F.tanh(model.conv(x).transpose(1,2))
print(f'Conv ouput shape => {x.shape}')
attn = F.softmax(model.U.weight.matmul(x.transpose (1,2)) , dim=2)
print(f'Attention Shape  => {attn.shape}')
m = attn.matmul(x)
print(f'Atten Mul shape  => {m.shape}')
y_hat = model.final.weight.mul(m).sum(dim=2).add(model.final.bias)
y_hat = (F.sigmoid(y_hat) > 0.5).long()
print(y_hat.shape)

X_train shape    => torch.Size([1, 2500])
Embedding Shpae  => torch.Size([1, 2500, 100])
Transpose Shape  => torch.Size([1, 100, 2500])
Conv ouput shape => torch.Size([1, 2500, 15])
Attention Shape  => torch.Size([1, 50, 2500])
Atten Mul shape  => torch.Size([1, 50, 15])
torch.Size([1, 50])


In [153]:
y_hat = y_hat.squeeze()
Y_samples = Y_samples.squeeze()
attn = attn.squeeze()

In [ ]:
# word_model.wv.index_to_key
doc = [word_model.wv.index_to_key[val.item()] for val in X_sample.flatten() if val < word_model.wv.vectors.shape[0]]
" ".join(doc)

'admission date discharge date date of birth sex m service history of the present illness this is a year old man with a history of genotype 3a hepatitis c alcohol abuse and cirrhosis chronic renal insufficiency hypertension with multiple hospitalizations secondary to complications of cirrhosis who presents with lethargy and with one episode of hematemesis the patient reports not feeling well for the past two to three days he describes feeling fatigued and sleepy yesterday the patient developed intermittent nausea headache and abdominal pain first name11 name pattern1 last name namepattern1 m d md number dictated by dictator info medquist36 d t job job number'

In [85]:
print(y_hat.type(torch.float32))
print(Y_samples)
np.where(Y_samples.squeeze()==1)

tensor([0., 0., 1., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 1., 1., 1., 1., 0., 0., 0.,
        1., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 1., 1.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


(array([ 9, 28, 35]),)

In [86]:
example     = 28
filter_size = 5
padding = int(float(filter_size//2)); print(f'Padding => {padding}')

Padding => 2


In [184]:
doc_sample = X_sample.squeeze().tolist()
doc_sample.insert(0,150853)
doc_sample.insert(0,150853)
doc_sample.extend([150853,150853])

In [150]:
windows = []
for idx in range(2,len(doc_sample)-2):
    # print(f'{doc_sample[idx-2:idx+3]}')
    windows.append(doc_sample[idx-2:idx+3])
print(len(windows))

2500


In [162]:
def translate(seq):
    doc = [word_model.wv.index_to_key[val] for val in seq if val < word_model.wv.vectors.shape[0]]
    return " ".join(doc)

In [166]:
count = 0
importance = torch.argsort(attn[example])
for imp in importance.tolist():
    if count == 10:
        break
    # print(windows[imp])
    print(translate(windows[imp]))
    count += 1

dictator info medquist36 d t
to complications of cirrhosis who
and abdominal pain first name11
of cirrhosis who presents with
pain first name11 name pattern1
not feeling well for the
of genotype 3a hepatitis c
past two to three days
one episode of hematemesis the
m d md number dictated


---

In [ ]:
# s = [0 for _ in range(5)]
# for idx in range(1,4): s[idx] += 1
# s

[0, 1, 1, 1, 0]

In [ ]:
word_scores = [0 for _ in doc_sample]
count = 0 
importance = attn[example]
for idx in range(len(doc_sample) - filter_size + 1):
    score = importance[idx]
    # print(doc_sample[idx:idx +filter_size])
    for i in range(idx,idx + filter_size):
        word_scores[i] += score.item()
    count += 1

In [204]:
words_weighted = list(zip(word_scores,doc_sample))
words_weighted = sorted(words_weighted,key = lambda x: x[0],reverse=True)

In [206]:
count = 0
for w,s in words_weighted:
    if count == 5:
        break
    word = word_model.wv.index_to_key[s]
    print(f'{word}=>{s}')
    count += 1

and=>1
cirrhosis=>1056
secondary=>240
abuse=>820
renal=>173
